# 04. Extra Trees Regression for Motor UPDRS Prediction

**Research question:** Does Extra Trees achieve a lower mean MAE than Ridge and the training-mean baseline when evaluated on the same three patient-wise folds?

## 1. Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

## 2. Load the Parkinson’s Telemonitoring Dataset

In [2]:
data_path = Path("../data/parkinsons_updrs.data")
df = pd.read_csv(data_path)

print("Recordings:", len(df))
print("Columns:", df.shape[1])
print("Unique patients:", df["subject#"].nunique())

Recordings: 5875
Columns: 22
Unique patients: 42


## 3. Define Voice Features, Target, and Patient Groups

In [3]:
voice_features = [
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE"
]

X = df[voice_features]
y = df["motor_UPDRS"]
groups = df["subject#"]

print("Input shape (recordings, features):", X.shape)
print("Target:", y.name)
print("Patient ID column:", groups.name)

Input shape (recordings, features): (5875, 16)
Target: motor_UPDRS
Patient ID column: subject#


## 4. Recreate and Verify the Original Patient Split

In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
groups_train = groups.iloc[train_idx]

train_ids = set(groups_train)
test_ids = set(groups.iloc[test_idx])
previous_test_ids = {5, 9, 14, 20, 26, 27, 30, 31, 40}

print("Training patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Patient overlap:", len(train_ids & test_ids))
print("Same test patients as Notebook 03:", test_ids == previous_test_ids)

Training patients: 33
Test patients: 9
Patient overlap: 0
Same test patients as Notebook 03: True


## 5. Recreate the Three Patient-wise Validation Folds

In [5]:
cv = GroupKFold(n_splits=3)
folds = list(cv.split(X_train, y_train, groups=groups_train))

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    fit_ids = set(groups_train.iloc[fit_idx])
    val_ids = set(groups_train.iloc[val_idx])

    print(f"Fold {fold}: training patients = {len(fit_ids)}, "
          f"validation patients = {len(val_ids)}, "
          f"overlap = {len(fit_ids & val_ids)}")

Fold 1: training patients = 22, validation patients = 11, overlap = 0
Fold 2: training patients = 22, validation patients = 11, overlap = 0
Fold 3: training patients = 22, validation patients = 11, overlap = 0


## 6. Fit Extra Trees on Fold 1

In [6]:
fit_idx, val_idx = folds[0]  # Fold 1

extra_trees_fold1 = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

extra_trees_fold1.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


## 7. Predict Motor UPDRS Scores on Fold 1

In [7]:
et_pred_fold1 = extra_trees_fold1.predict(X_train.iloc[val_idx])

preview = pd.DataFrame({
    "reference_motor_UPDRS": y_train.iloc[val_idx],
    "extra_trees_prediction": et_pred_fold1
})

preview.head(5).round(4)

,reference_motor_UPDRS,extra_trees_prediction
438,10.737,24.1776
439,11.270,21.5096
440,11.803,20.8979
441,12.338,24.1058
442,12.941,27.7393


## 8. Evaluate Extra Trees on Fold 1

In [8]:
et_mae_fold1 = mean_absolute_error(y_train.iloc[val_idx], et_pred_fold1)
print(f"Fold 1 Extra Trees MAE: {et_mae_fold1:.4f} points")

Fold 1 Extra Trees MAE: 7.5672 points


**Observation:** On Fold 1, Extra Trees had a higher MAE (7.5672 points) than both Ridge (7.0022) and the training-mean baseline (6.9911).

## 9. Evaluate Extra Trees Across All Three Folds

In [9]:
et_fold_results = []

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    model = ExtraTreesRegressor(n_estimators=100, random_state=42)
    model.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])

    predictions = model.predict(X_train.iloc[val_idx])
    mae = mean_absolute_error(y_train.iloc[val_idx], predictions)

    et_fold_results.append({"fold": fold, "extra_trees_MAE": mae})

et_cv_results = pd.DataFrame(et_fold_results)
et_cv_results.round(4)

,fold,extra_trees_MAE
0,1,7.5672
1,2,8.6373
2,3,8.6090


**Observation:** Across the three patient-wise folds, Extra Trees had a mean MAE of 8.2712 points, compared with 8.1843 for Ridge and 7.2706 for the training-mean baseline. Extra Trees outperformed Ridge only in Fold 3 and did not outperform the baseline in any fold.

## 10. Save the Fold-wise Model Comparison

In [10]:
results_dir = Path("../results")

previous_results = pd.read_csv(results_dir / "patient_wise_cv_fold_results.csv")
comparison = previous_results[["fold", "baseline_MAE", "ridge_MAE"]].merge(
    et_cv_results,
    on="fold",
    validate="one_to_one"
).sort_values("fold")

comparison.to_csv(results_dir / "patient_wise_model_comparison.csv", index=False)
comparison.round(4)

,fold,baseline_MAE,ridge_MAE,extra_trees_MAE
0,1,6.9911,7.0022,7.5672
1,2,8.1967,7.7026,8.6373
2,3,6.6242,9.8480,8.6090


## 11. Save the Mean MAE Across Folds

In [11]:
summary = (
    comparison[["baseline_MAE", "ridge_MAE", "extra_trees_MAE"]]
    .mean()
    .rename_axis("metric")
    .reset_index(name="mean_fold_MAE")
    .sort_values("mean_fold_MAE")
)

summary.to_csv(results_dir / "patient_wise_model_summary.csv", index=False)
summary.round(4)

,metric,mean_fold_MAE
0,baseline_MAE,7.2706
1,ridge_MAE,8.1843
2,extra_trees_MAE,8.2712
